# 04 - ImageNet-100 Pipeline Test

Test the self-healing pipeline:
- Compare: Clean vs Noisy vs Healed accuracy
- Test multiple noise levels
- Visualize healing quality
- Generate benchmark table

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

# Configuration
BASE_DIR = r"C:\Users\Rushikesh\OneDrive\CODES\SelfHealingNN"
os.chdir(BASE_DIR)

torch.set_num_threads(8)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 100

print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Models

In [ ]:
# VAE Architecture (same as training)
class ImageNet100VAE(nn.Module):
    def __init__(self, latent_dim=256):
        super(ImageNet100VAE, self).__init__()
        self.latent_dim = latent_dim
        
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 3, stride=2, padding=1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
        )
        
        self.flatten_size = 512 * 7 * 7
        self.fc_mu = nn.Linear(self.flatten_size, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_size, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, self.flatten_size)
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std
    
    def forward(self, x):
        h = self.encoder(x).view(-1, self.flatten_size)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        h2 = F.relu(self.fc_decode(z)).view(-1, 512, 7, 7)
        return self.decoder(h2), mu, logvar


# Expert Architecture
class ImageNet100Expert(nn.Module):
    def __init__(self, num_classes=100, dropout=0.3):
        super(ImageNet100Expert, self).__init__()
        self.resnet = models.resnet18(weights=None)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )
    
    def forward(self, x):
        return self.resnet(x)

In [ ]:
# Load trained models
healer = ImageNet100VAE(latent_dim=256).to(device)
healer.load_state_dict(torch.load(os.path.join(BASE_DIR, "models", "imagenet100_healer.pth")))
healer.eval()

expert = ImageNet100Expert(num_classes=NUM_CLASSES).to(device)
expert.load_state_dict(torch.load(os.path.join(BASE_DIR, "models", "imagenet100_expert.pth")))
expert.eval()

print("Models loaded successfully!")
print(f"  Healer: ImageNet100VAE")
print(f"  Expert: ImageNet100Expert (ResNet-18)")

## 2. Load Data

In [ ]:
# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Transforms WITHOUT normalization (for VAE pipeline)
val_transforms_raw = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

# Load validation data
DATA_DIR = os.path.join(BASE_DIR, "imagenet100_data")
VAL_DIR = os.path.join(DATA_DIR, "val")

val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_transforms_raw)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

class_names = val_dataset.classes
print(f"Validation set: {len(val_dataset)} images")

## 3. Define Noise and Normalization Functions

In [ ]:
def add_noise(image_tensor, noise_factor=0.3):
    """Add Gaussian noise."""
    noise = torch.randn_like(image_tensor) * noise_factor
    return torch.clamp(image_tensor + noise, 0., 1.)


def normalize_for_resnet(tensor):
    """Normalize tensor for ResNet (ImageNet stats)."""
    mean = torch.tensor(IMAGENET_MEAN, device=tensor.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=tensor.device).view(1, 3, 1, 1)
    return (tensor - mean) / std


def denormalize(tensor):
    """Denormalize for visualization."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return tensor.cpu() * std + mean

print("Helper functions defined")

## 4. Evaluate Pipeline

In [ ]:
def evaluate_pipeline(healer, expert, loader, noise_factor, device):
    """
    Evaluate the full pipeline: Clean vs Noisy vs Healed.
    
    Returns:
        dict with clean_acc, noisy_acc, healed_acc (top-1 and top-5)
    """
    healer.eval()
    expert.eval()
    
    clean_correct = clean_correct5 = 0
    noisy_correct = noisy_correct5 = 0
    healed_correct = healed_correct5 = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            batch_size = images.size(0)
            total += batch_size
            
            # Clean images
            clean_norm = normalize_for_resnet(images)
            clean_out = expert(clean_norm)
            _, clean_pred = clean_out.max(1)
            _, clean_top5 = clean_out.topk(5, 1)
            clean_correct += clean_pred.eq(labels).sum().item()
            clean_correct5 += clean_top5.eq(labels.view(-1, 1)).sum().item()
            
            # Noisy images (direct to expert)
            noisy = add_noise(images, noise_factor)
            noisy_norm = normalize_for_resnet(noisy)
            noisy_out = expert(noisy_norm)
            _, noisy_pred = noisy_out.max(1)
            _, noisy_top5 = noisy_out.topk(5, 1)
            noisy_correct += noisy_pred.eq(labels).sum().item()
            noisy_correct5 += noisy_top5.eq(labels.view(-1, 1)).sum().item()
            
            # Healed images (noisy -> VAE -> expert)
            healed, _, _ = healer(noisy)
            healed_norm = normalize_for_resnet(healed)
            healed_out = expert(healed_norm)
            _, healed_pred = healed_out.max(1)
            _, healed_top5 = healed_out.topk(5, 1)
            healed_correct += healed_pred.eq(labels).sum().item()
            healed_correct5 += healed_top5.eq(labels.view(-1, 1)).sum().item()
    
    return {
        'clean_top1': 100. * clean_correct / total,
        'clean_top5': 100. * clean_correct5 / total,
        'noisy_top1': 100. * noisy_correct / total,
        'noisy_top5': 100. * noisy_correct5 / total,
        'healed_top1': 100. * healed_correct / total,
        'healed_top5': 100. * healed_correct5 / total,
    }

print("Evaluation function ready")

## 5. Test Multiple Noise Levels

In [ ]:
noise_levels = [0.1, 0.2, 0.3, 0.4, 0.5]
results = []

print("Testing different noise levels...")
print("="*70)

for noise in noise_levels:
    print(f"\nEvaluating noise level: {noise}")
    metrics = evaluate_pipeline(healer, expert, val_loader, noise, device)
    
    recovery = metrics['healed_top1'] - metrics['noisy_top1']
    
    print(f"  Clean:  Top-1: {metrics['clean_top1']:.1f}%  Top-5: {metrics['clean_top5']:.1f}%")
    print(f"  Noisy:  Top-1: {metrics['noisy_top1']:.1f}%  Top-5: {metrics['noisy_top5']:.1f}%")
    print(f"  Healed: Top-1: {metrics['healed_top1']:.1f}%  Top-5: {metrics['healed_top5']:.1f}%")
    print(f"  Recovery: +{recovery:.1f}%")
    
    results.append({
        'Noise': noise,
        'Clean Top-1': f"{metrics['clean_top1']:.1f}%",
        'Noisy Top-1': f"{metrics['noisy_top1']:.1f}%",
        'Healed Top-1': f"{metrics['healed_top1']:.1f}%",
        'Recovery': f"+{recovery:.1f}%",
        'Clean Top-5': f"{metrics['clean_top5']:.1f}%",
        'Noisy Top-5': f"{metrics['noisy_top5']:.1f}%",
        'Healed Top-5': f"{metrics['healed_top5']:.1f}%",
    })

print("\n" + "="*70)

## 6. Results Table

In [ ]:
df = pd.DataFrame(results)
print("\nBenchmark Results:")
print(df.to_string(index=False))

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Extract numeric values
clean_acc = [float(r['Clean Top-1'].replace('%', '')) for r in results]
noisy_acc = [float(r['Noisy Top-1'].replace('%', '')) for r in results]
healed_acc = [float(r['Healed Top-1'].replace('%', '')) for r in results]

# Top-1 Accuracy
x = np.arange(len(noise_levels))
width = 0.25

axes[0].bar(x - width, clean_acc, width, label='Clean', color='green', alpha=0.8)
axes[0].bar(x, noisy_acc, width, label='Noisy', color='red', alpha=0.8)
axes[0].bar(x + width, healed_acc, width, label='Healed', color='blue', alpha=0.8)

axes[0].set_xlabel('Noise Level')
axes[0].set_ylabel('Top-1 Accuracy (%)')
axes[0].set_title('Top-1 Accuracy: Clean vs Noisy vs Healed')
axes[0].set_xticks(x)
axes[0].set_xticklabels(noise_levels)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Recovery
recovery = [h - n for h, n in zip(healed_acc, noisy_acc)]
colors = ['green' if r > 0 else 'red' for r in recovery]
axes[1].bar(noise_levels, recovery, color=colors, alpha=0.8, width=0.06)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Noise Level')
axes[1].set_ylabel('Recovery (%)')
axes[1].set_title('Accuracy Recovery (Healed - Noisy)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Visualize Healing Quality

In [ ]:
# Get sample images
images, labels = next(iter(val_loader))
images = images.to(device)

# Test with noise factor 0.4
noise_factor = 0.4
noisy = add_noise(images, noise_factor)

with torch.no_grad():
    healed, _, _ = healer(noisy)

# Visualize
fig, axes = plt.subplots(3, 6, figsize=(18, 9))

for col in range(6):
    # Original
    img = images[col].cpu().numpy().transpose(1, 2, 0)
    axes[0, col].imshow(np.clip(img, 0, 1))
    axes[0, col].set_title(f"{class_names[labels[col]][:15]}")
    axes[0, col].axis('off')
    
    # Noisy
    img = noisy[col].cpu().numpy().transpose(1, 2, 0)
    axes[1, col].imshow(np.clip(img, 0, 1))
    axes[1, col].axis('off')
    
    # Healed
    img = healed[col].cpu().numpy().transpose(1, 2, 0)
    axes[2, col].imshow(np.clip(img, 0, 1))
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12, rotation=0, labelpad=50)
axes[1, 0].set_ylabel('Noisy (0.4)', fontsize=12, rotation=0, labelpad=50)
axes[2, 0].set_ylabel('Healed', fontsize=12, rotation=0, labelpad=50)

plt.suptitle(f"Self-Healing Pipeline (Noise: {noise_factor})", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
# Best result (typically at noise 0.3-0.4)
best_idx = 2  # noise 0.3
best_noise = noise_levels[best_idx]
best_clean = clean_acc[best_idx]
best_noisy = noisy_acc[best_idx]
best_healed = healed_acc[best_idx]
best_recovery = best_healed - best_noisy

print("="*60)
print("Self-Healing Neural Network - Pipeline Test Results")
print("="*60)
print(f"\nDataset: ImageNet-100 (100 classes, ~130k images)")
print(f"Healer: ImageNet100VAE")
print(f"Expert: ResNet-18 (pretrained)")
print(f"")
print(f"Key Results (Noise={best_noise}):")
print(f"  Clean accuracy:  {best_clean:.1f}%")
print(f"  Noisy accuracy:  {best_noisy:.1f}%")
print(f"  Healed accuracy: {best_healed:.1f}%")
print(f"  Recovery:        +{best_recovery:.1f}%")
print(f"")
print("="*60)
print("\nNext: Run 05_ImageNet100_End_to_End.ipynb for joint fine-tuning!")